# Notebook de Producción — Predicción de CO₂ en Planta de Acero DAEWOO

**Objetivo**: Pipeline de producción listo para consumo por la aplicación Streamlit.
**Dataset**: Planta de Acero DAEWOO, Corea del Sur, 2018. Mediciones cada 15 minutos.
**Target**: `CO2_tCO2` — emisiones de CO₂ en toneladas por intervalo de 15 minutos.

## Sección 1 — Imports y Rutas

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import json
from pathlib import Path
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Raíz del repositorio (patrón obligatorio)
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "README.md").exists():
    repo_root = repo_root.parent
if not (repo_root / "README.md").exists():
    raise FileNotFoundError("No se encontró la raíz del repositorio.")

print(f"Repositorio raíz: {repo_root}")

Repositorio raíz: C:\Users\matia\GitHub\Energy_consumption


## Sección 2 — Decisión del Modelo de Producción

### Evaluación de los dos experimentos

| Experimento | R² validación | RMSE validación | Variables clave |
|---|---|---|---|
| A | 0.9959 | 0.0644 (estandarizado) | `Usage_kWh`, `Lagging_Current_Reactive_Power_kVarh` |
| B | 0.8745 | 0.3557 (estandarizado) | Temporales + eficiencia eléctrica |

### ¿Por qué **Experimento B** es el modelo de producción?

El **Experimento A** alcanza R²=0.9959, rendimiento casi perfecto. Sin embargo, su variable más
predictiva es `Usage_kWh` (correlación con CO₂: r=0.988). Esta variable representa el consumo
eléctrico total del intervalo actual — el mismo que *genera* las emisiones que queremos predecir.
En una aplicación real, el operador no conoce `Usage_kWh` antes de que ocurra el consumo:
usarla como input es casi tautológico y destruye la capacidad de anticipación.

El **Experimento B** usa exclusivamente:
- **Variables temporales**: hora del día, turno (mañana/tarde/noche), día de semana, NSM codificado
  como seno y coseno (ciclicidad del día).
- **Variables de eficiencia eléctrica**: factores de potencia y potencia reactiva leading, que el
  operador puede ajustar o conocer antes de que ocurra el consumo.

Con R²=0.8745 en datos no vistos, el modelo B ofrece:
- **Anticipación**: el operador puede ingresar el turno y tipo de carga *planificados* y obtener una
  estimación de CO₂ esperado antes de que empiece el período.
- **Accionabilidad**: las variables de entrada son ajustables operacionalmente (redistribución de
  carga, ajuste de turno, mantenimiento programado).
- **Interpretabilidad**: el operador entiende qué significa "turno de noche" o "carga máxima".

### Rol del Experimento A como modelo de referencia

El Experimento A se conserva como **modelo de auditoría/monitoreo** en contextos donde
`Usage_kWh` ya está disponible (p. ej., análisis post-hoc, detección de anomalías en tiempo real
durante el turno). Su limitación es explícita: no puede anticipar consumo futuro.

**Conclusión**: producción usa **Exp B** como modelo principal; Exp A como referencia.

## Sección 3 — Cargar Artefactos

In [2]:
hist_dir = repo_root / "03_Modelos" / "01_Historial"

# ── Experimento B (modelo de producción) ──────────────────────────────────────
encoder_B = joblib.load(hist_dir / "pipeline_encoder_B.joblib")
scaler_B  = joblib.load(hist_dir / "pipeline_scaler_B.joblib")
modelo_B  = joblib.load(hist_dir / "rfc_CO2_B_v1_pipeline.joblib")

# ── Experimento A (modelo de referencia) ──────────────────────────────────────
encoder_A = joblib.load(hist_dir / "pipeline_encoder_A.joblib")
scaler_A  = joblib.load(hist_dir / "pipeline_scaler_A.joblib")
modelo_A  = joblib.load(hist_dir / "rfc_CO2_A_v1_pipeline.joblib")

# Verificación de artefactos B
print("=== Artefactos Exp B (Producción) ===")
print(f"encoder_B tipo: {type(encoder_B).__name__}")
print(f"encoder_B features: {list(encoder_B.feature_names_in_)}")
print(f"encoder_B categorías: {[list(c) for c in encoder_B.categories_]}")
print(f"scaler_B tipo: {type(scaler_B).__name__}")
print(f"scaler_B n_features: {scaler_B.n_features_in_}")
print(f"scaler_B mean (CO2, idx 6): {scaler_B.mean_[6]:.8f}")
print(f"scaler_B scale (CO2, idx 6): {scaler_B.scale_[6]:.8f}")
print(f"modelo_B tipo: {type(modelo_B).__name__}")
print(f"modelo_B n_features_in: {modelo_B.n_features_in_}")
print()
print("=== Artefactos Exp A (Referencia) ===")
print(f"encoder_A tipo: {type(encoder_A).__name__}")
print(f"encoder_A features: {list(encoder_A.feature_names_in_)}")
print(f"scaler_A tipo: {type(scaler_A).__name__}")
print(f"modelo_A tipo: {type(modelo_A).__name__}")
print(f"modelo_A n_features_in: {modelo_A.n_features_in_}")

=== Artefactos Exp B (Producción) ===
encoder_B tipo: OneHotEncoder
encoder_B features: ['Day_of_week', 'Load_Type', 'turno']
encoder_B categorías: [['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday'], ['Light_Load', 'Maximum_Load', 'Medium_Load'], ['Mañana', 'Noche', 'Tarde']]
scaler_B tipo: StandardScaler
scaler_B n_features: 7
scaler_B mean (CO2, idx 6): 0.01152426
scaler_B scale (CO2, idx 6): 0.01615059
modelo_B tipo: Pipeline
modelo_B n_features_in: 20

=== Artefactos Exp A (Referencia) ===
encoder_A tipo: OneHotEncoder
encoder_A features: ['WeekStatus', 'Day_of_week', 'Load_Type']
scaler_A tipo: StandardScaler
modelo_A tipo: RandomForestRegressor
modelo_A n_features_in: 18


## Sección 4 — Función de Preprocesamiento

In [3]:
def preprocesar_input(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Preprocesa un DataFrame raw con columnas del CSV original y devuelve
    X listo para modelo_B.predict().

    Parámetros esperados en df_raw:
        - date: datetime o string parseable (formato dayfirst=True)
        - Leading_Current_Reactive_Power_kVarh
        - Lagging_Current_Power_Factor
        - Leading_Current_Power_Factor
        - NSM
        - Day_of_week
        - Load_Type
        - WeekStatus

    Columnas opcionales con nombres alternativos del CSV:
        - 'CO2(tCO2)' → se renombra a CO2_tCO2
        - 'Lagging_Current_Reactive.Power_kVarh' → se renombra

    Retorna:
        pd.DataFrame X con las 20 features esperadas por modelo_B (sin target).
    """
    df = df_raw.copy()

    # 1. Renombrar columnas inconsistentes del CSV original
    df = df.rename(columns={
        "CO2(tCO2)":                           "CO2_tCO2",
        "Lagging_Current_Reactive.Power_kVarh": "Lagging_Current_Reactive_Power_kVarh",
    })

    # 2. Convertir fecha
    if not pd.api.types.is_datetime64_any_dtype(df["date"]):
        df["date"] = pd.to_datetime(df["date"], dayfirst=True)

    # 3. Features temporales
    df["hora"] = df["date"].dt.hour
    df["turno"] = df["hora"].map(
        lambda h: "Mañana" if 6 <= h < 14 else ("Tarde" if 14 <= h < 22 else "Noche")
    )
    df["es_fin_de_semana"] = (df["WeekStatus"] == "Weekend").astype(int)
    df["NSM_sin"] = np.sin(2 * np.pi * df["NSM"] / 86400)
    df["NSM_cos"] = np.cos(2 * np.pi * df["NSM"] / 86400)

    # 4. OHE sobre categóricas (mismo orden que entrenamiento)
    cat_B = df[["Day_of_week", "Load_Type", "turno"]]
    cat_ohe_arr = encoder_B.transform(cat_B)
    cat_ohe_cols = encoder_B.get_feature_names_out()
    cat_ohe = pd.DataFrame(cat_ohe_arr, columns=cat_ohe_cols, index=df.index)

    # 5. Escalar numéricas
    # El scaler_B fue entrenado con 7 columnas; CO2_tCO2 es la columna 6 (índice final).
    # En inferencia CO2_tCO2 no está disponible → rellenar con ceros y descartar columna 6.
    num_input = np.column_stack([
        df["Leading_Current_Reactive_Power_kVarh"],
        df["Lagging_Current_Power_Factor"],
        df["Leading_Current_Power_Factor"],
        df["NSM_sin"],
        df["NSM_cos"],
        df["hora"],
        np.zeros(len(df)),  # placeholder para CO2_tCO2 (no disponible en inferencia)
    ])
    num_scaled_full = scaler_B.transform(num_input)
    num_scaled = num_scaled_full[:, :6]  # descartar columna 6 (CO2_tCO2)

    num_feature_names = [
        "Leading_Current_Reactive_Power_kVarh_std_",
        "Lagging_Current_Power_Factor_std_",
        "Leading_Current_Power_Factor_std_",
        "NSM_sin_std_",
        "NSM_cos_std_",
        "hora_std_",
    ]
    num_df = pd.DataFrame(num_scaled, columns=num_feature_names, index=df.index)

    # 6. es_fin_de_semana como columna independiente
    es_fin_col = df[["es_fin_de_semana"]].copy()

    # 7. Ensamblar X final: [cat_ohe | es_fin_de_semana | num_scaled]
    X = pd.concat([cat_ohe, es_fin_col, num_df], axis=1)

    return X


# Prueba rápida de la función con 3 registros de validación
_df_test = pd.read_csv(repo_root / "01_Datos" / "02_Validacion" / "validacion.csv", nrows=3)
_X_test = preprocesar_input(_df_test)
print(f"preprocesar_input OK — shape: {_X_test.shape}")
print(f"Columnas: {_X_test.columns.tolist()}")

preprocesar_input OK — shape: (3, 20)
Columnas: ['Day_of_week_Friday', 'Day_of_week_Monday', 'Day_of_week_Saturday', 'Day_of_week_Sunday', 'Day_of_week_Thursday', 'Day_of_week_Tuesday', 'Day_of_week_Wednesday', 'Load_Type_Light_Load', 'Load_Type_Maximum_Load', 'Load_Type_Medium_Load', 'turno_Mañana', 'turno_Noche', 'turno_Tarde', 'es_fin_de_semana', 'Leading_Current_Reactive_Power_kVarh_std_', 'Lagging_Current_Power_Factor_std_', 'Leading_Current_Power_Factor_std_', 'NSM_sin_std_', 'NSM_cos_std_', 'hora_std_']


## Sección 5 — Función de Predicción

In [4]:
def predecir_co2(df_raw: pd.DataFrame, retornar_escala_original: bool = True) -> np.ndarray:
    """
    Predice emisiones de CO₂ a partir de variables operacionales.

    Parámetros:
        df_raw: DataFrame con columnas raw (ver preprocesar_input).
        retornar_escala_original: si True, convierte de escala estandarizada a tCO2.
            Usa los parámetros del scaler_B: mean_[6] y scale_[6].

    Retorna:
        np.ndarray de predicciones en tCO2 (o en escala estandarizada si False).
    """
    X = preprocesar_input(df_raw)
    y_pred_std = modelo_B.predict(X)

    if retornar_escala_original:
        # Inverse-transform: y_original = y_std * scale + mean
        return y_pred_std * scaler_B.scale_[6] + scaler_B.mean_[6]

    return y_pred_std


# Prueba con 5 registros
_df_demo = pd.read_csv(repo_root / "01_Datos" / "02_Validacion" / "validacion.csv", nrows=5)
_preds = predecir_co2(_df_demo)
print(f"predecir_co2 OK — shape: {_preds.shape}")
print(f"Primeras predicciones (tCO2): {_preds}")
print(f"scaler_B.mean_[6] (media CO2): {scaler_B.mean_[6]:.8f}")
print(f"scaler_B.scale_[6] (std CO2): {scaler_B.scale_[6]:.8f}")

predecir_co2 OK — shape: (5,)
Primeras predicciones (tCO2): [-2.28983499e-16  2.49410112e-02  4.35341243e-02  3.46944695e-18
  3.98618470e-02]
scaler_B.mean_[6] (media CO2): 0.01152426
scaler_B.scale_[6] (std CO2): 0.01615059


## Sección 6 — Validación del Pipeline End-to-End

In [5]:
# Cargar dataset de validación completo
val_path = repo_root / "01_Datos" / "02_Validacion" / "validacion.csv"
df_val = pd.read_csv(val_path)
print(f"Dataset de validación cargado: {df_val.shape[0]} registros, {df_val.shape[1]} columnas")

# Renombrar columnas inconsistentes para extraer el target
df_val = df_val.rename(columns={
    "CO2(tCO2)":                            "CO2_tCO2",
    "Lagging_Current_Reactive.Power_kVarh": "Lagging_Current_Reactive_Power_kVarh",
})

# Aplicar pipeline de producción (sin tocar el target)
y_pred_orig = predecir_co2(df_val)   # predicciones en tCO2
y_real_orig = df_val["CO2_tCO2"].values  # valores reales en tCO2

# Métricas en escala original (tCO2)
r2_orig   = r2_score(y_real_orig, y_pred_orig)
rmse_orig = np.sqrt(mean_squared_error(y_real_orig, y_pred_orig))
mae_orig  = mean_absolute_error(y_real_orig, y_pred_orig)

print()
print("=== MÉTRICAS VALIDACIÓN — ESCALA ORIGINAL (tCO₂) ===")
print(f"R²     : {r2_orig:.6f}")
print(f"RMSE   : {rmse_orig:.6f} tCO₂")
print(f"MAE    : {mae_orig:.6f} tCO₂")
print()
print("Nota: R² idéntico al de preproducción (escala estandarizada) — la transformación")
print("lineal preserva la correlación. RMSE y MAE están ahora en unidades reales (tCO₂).")

# Guardar métricas para usar en Sección 7 y en reporte JSON
metricas_val_B_orig = {
    "r2":       round(float(r2_orig),   6),
    "rmse_tco2": round(float(rmse_orig), 6),
    "mae_tco2":  round(float(mae_orig),  6),
}
print()
print("Métricas guardadas en variable metricas_val_B_orig:")
print(metricas_val_B_orig)

Dataset de validación cargado: 10512 registros, 11 columnas

=== MÉTRICAS VALIDACIÓN — ESCALA ORIGINAL (tCO₂) ===
R²     : 0.874477
RMSE   : 0.005745 tCO₂
MAE    : 0.002797 tCO₂

Nota: R² idéntico al de preproducción (escala estandarizada) — la transformación
lineal preserva la correlación. RMSE y MAE están ahora en unidades reales (tCO₂).

Métricas guardadas en variable metricas_val_B_orig:
{'r2': 0.874477, 'rmse_tco2': 0.005745, 'mae_tco2': 0.002797}


## Sección 7 — Guardar Pipeline de Producción

In [6]:
prod_dir = repo_root / "03_Modelos" / "02_Produccion"
prod_dir.mkdir(parents=True, exist_ok=True)

# ── Pipeline de producción (Exp B) ────────────────────────────────────────────
pipeline_produccion = {
    "encoder":  encoder_B,
    "scaler":   scaler_B,
    "modelo":   modelo_B,
    "experimento": "B",
    "features_input": [
        "Day_of_week", "Load_Type", "turno", "es_fin_de_semana",
        "Leading_Current_Reactive_Power_kVarh",
        "Lagging_Current_Power_Factor",
        "Leading_Current_Power_Factor",
        "NSM_sin", "NSM_cos", "hora",
    ],
    "target": "CO2_tCO2",
    "metricas_validacion": {
        "r2":      0.874477,
        "rmse":    0.355691,
        "mae":     0.173210,
    },
}

path_prod = prod_dir / "pipeline_produccion_B_v1.joblib"
joblib.dump(pipeline_produccion, path_prod)
assert path_prod.exists(), "ERROR: pipeline de producción no guardado"
print(f"Pipeline de producción guardado: {path_prod}")

# Verificar que se puede recargar
_pp = joblib.load(path_prod)
assert "modelo" in _pp and "encoder" in _pp and "scaler" in _pp
print(f"  → Verificado: keys={list(_pp.keys())}")

# ── Pipeline de referencia (Exp A) ────────────────────────────────────────────
pipeline_referencia = {
    "encoder":  encoder_A,
    "scaler":   scaler_A,
    "modelo":   modelo_A,
    "experimento": "A",
    "nota": (
        "Modelo de referencia para auditoría. "
        "Requiere Usage_kWh en tiempo real. "
        "No apto para predicción anticipada — Usage_kWh es consecuencia del consumo a predecir."
    ),
    "metricas_validacion": {
        "r2":   0.995889,
        "rmse": 0.064370,
        "mae":  0.004801,
    },
}

path_ref = prod_dir / "pipeline_referencia_A_v1.joblib"
joblib.dump(pipeline_referencia, path_ref)
assert path_ref.exists(), "ERROR: pipeline de referencia no guardado"
print(f"Pipeline de referencia guardado: {path_ref}")

_pr = joblib.load(path_ref)
assert "modelo" in _pr
print(f"  → Verificado: keys={list(_pr.keys())}")

Pipeline de producción guardado: C:\Users\matia\GitHub\Energy_consumption\03_Modelos\02_Produccion\pipeline_produccion_B_v1.joblib


  → Verificado: keys=['encoder', 'scaler', 'modelo', 'experimento', 'features_input', 'target', 'metricas_validacion']
Pipeline de referencia guardado: C:\Users\matia\GitHub\Energy_consumption\03_Modelos\02_Produccion\pipeline_referencia_A_v1.joblib
  → Verificado: keys=['encoder', 'scaler', 'modelo', 'experimento', 'nota', 'metricas_validacion']


In [7]:
# ── Reporte de producción JSON ────────────────────────────────────────────────
result_dir = repo_root / "04_Resultados" / "03_Produccion"
result_dir.mkdir(parents=True, exist_ok=True)

reporte = {
    "modelo_produccion": "B",
    "justificacion": (
        "El Experimento B es el modelo de producción porque sus variables de entrada "
        "(hora del día, turno, día de semana, factores de potencia, potencia reactiva leading) "
        "son conocidas por el operador ANTES de que ocurra el consumo eléctrico. "
        "Esto permite anticipar emisiones de CO2 en escenarios de planificación. "
        "El Experimento A tiene mayor precisión (R2=0.9959) pero requiere Usage_kWh como input, "
        "variable que solo se conoce DESPUÉS de que ocurre el consumo — uso tautológico."
    ),
    "modelo_referencia": "A",
    "metricas_validacion_B": {
        "r2":       0.874477,
        "rmse_std": 0.355691,
        "mae_std":  0.173210,
    },
    "metricas_validacion_A": {
        "r2":       0.995889,
        "rmse_std": 0.064370,
        "mae_std":  0.004801,
    },
    "metricas_validacion_B_escala_original": metricas_val_B_orig,
    "features_produccion": [
        "Day_of_week", "Load_Type", "turno", "es_fin_de_semana",
        "Leading_Current_Reactive_Power_kVarh",
        "Lagging_Current_Power_Factor",
        "Leading_Current_Power_Factor",
        "NSM_sin", "NSM_cos", "hora",
    ],
    "artefactos": {
        "pipeline_produccion": "03_Modelos/02_Produccion/pipeline_produccion_B_v1.joblib",
        "pipeline_referencia": "03_Modelos/02_Produccion/pipeline_referencia_A_v1.joblib",
    },
    "limitaciones": [
        (
            "El modelo B alcanza R2=0.875 sin variables de proceso productivo "
            "(temperatura, toneladas). Para mayor precisión se requieren datos adicionales "
            "de la línea de producción."
        ),
        (
            "El dataset cubre solo 2018. La validez en otros períodos depende de la "
            "estabilidad de los patrones operacionales."
        ),
    ],
}

json_path = result_dir / "reporte_produccion.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(reporte, f, indent=2, ensure_ascii=False)

assert json_path.exists(), "ERROR: reporte JSON no guardado"
print(f"Reporte JSON guardado: {json_path}")

# Mostrar contenido
with open(json_path, encoding="utf-8") as f:
    contenido = json.load(f)
print()
print(json.dumps(contenido, indent=2, ensure_ascii=False))

Reporte JSON guardado: C:\Users\matia\GitHub\Energy_consumption\04_Resultados\03_Produccion\reporte_produccion.json

{
  "modelo_produccion": "B",
  "justificacion": "El Experimento B es el modelo de producción porque sus variables de entrada (hora del día, turno, día de semana, factores de potencia, potencia reactiva leading) son conocidas por el operador ANTES de que ocurra el consumo eléctrico. Esto permite anticipar emisiones de CO2 en escenarios de planificación. El Experimento A tiene mayor precisión (R2=0.9959) pero requiere Usage_kWh como input, variable que solo se conoce DESPUÉS de que ocurre el consumo — uso tautológico.",
  "modelo_referencia": "A",
  "metricas_validacion_B": {
    "r2": 0.874477,
    "rmse_std": 0.355691,
    "mae_std": 0.17321
  },
  "metricas_validacion_A": {
    "r2": 0.995889,
    "rmse_std": 0.06437,
    "mae_std": 0.004801
  },
  "metricas_validacion_B_escala_original": {
    "r2": 0.874477,
    "rmse_tco2": 0.005745,
    "mae_tco2": 0.002797
  },
  

## Resumen Final

### Archivos generados

| Archivo | Descripción |
|---|---|
| `03_Modelos/02_Produccion/pipeline_produccion_B_v1.joblib` | Pipeline principal (encoder_B + scaler_B + modelo_B) |
| `03_Modelos/02_Produccion/pipeline_referencia_A_v1.joblib` | Pipeline de auditoría (encoder_A + scaler_A + modelo_A) |
| `04_Resultados/03_Produccion/reporte_produccion.json` | Reporte con métricas y justificación |

### Métricas en escala original (tCO₂)

El R² se conserva idéntico al reportado en preproducción (escala estandarizada = transformación
lineal). RMSE y MAE ahora en unidades reales:

- **R²**: 0.8745
- **RMSE**: ~0.005745 tCO₂ por intervalo de 15 min
- **MAE**: ~0.002797 tCO₂ por intervalo de 15 min

### Uso desde la aplicación Streamlit

```python
import joblib
from pathlib import Path

repo_root = ...  # raíz del repositorio
pp = joblib.load(repo_root / "03_Modelos/02_Produccion/pipeline_produccion_B_v1.joblib")
encoder = pp["encoder"]
scaler  = pp["scaler"]
modelo  = pp["modelo"]

# O bien, usar directamente las funciones definidas en este notebook
predicciones = predecir_co2(df_nuevos_datos)
```